In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

plt.rcParams['animation.embed_limit'] = 30.0

N = 100 # the number of particles
D = 10 # size of container (box/cube), arbitrary units
dim = 3 # dimensions
ts = 300 # timesteps
vp = 0.1 # "max" particle speed, units/second ("max" as in this is what is used to get the scale for speed distribution)
# PV = nRT -> N_A*N = n moles (N_A is avocado's number), V = D^dim, but speed...
# for the speed at vp in x or y (or z), the max speed would be the magnitude, or
sig = np.sqrt(dim*(vp*vp)) # this is the scale for which we'll setup velocities by

# to initialize positions/velocities
rs = np.random.rand(N, dim) * D # positions, in a distribution ranging [0, 1]
vs = np.random.normal(loc = 0.0, scale = sig, size = (N, dim)) # velocities, in a normal distribution around the mean (loc) for each coordinate, leading to an MB speed distribution!

m = 1.0 # particle mass, used to track T later
kb = 1.0 # Boltzmann constant in simulation units, to get KE (equipartition) and then T

In [ ]:
# to make a plot
if dim == 2:
    fig, ax = plt.subplots()
    scatter = ax.scatter(rs[:, 0], rs[:, 1], marker = 'o') # particles are represented as dots, scattering the x and y positions
    ax.set_xlim(0, D)
    ax.set_ylim(0, D)
    intxt = ax.text(0.02, 0.95, "", transform=ax.transAxes, fontsize=10, 
                      color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
if dim == 3:
    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')
    scatter = ax.scatter(*rs.transpose(), marker = 'o') 
    ax.set_xlim(0, D)
    ax.set_ylim(0, D)
    ax.set_zlim(0, D)
    intxt = ax.text2D(0.02, 0.95, "", transform=ax.transAxes, fontsize=10, 
                      color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
plt.gca().set_aspect('equal', adjustable = 'box')

plt.show()

In [ ]:
# now for the iterations
colls = []
def update(frame):
    global rs, vs, colls 
    rs += vs
    collnum = 0
    for i in range(N):
        for j in range(dim):
            if rs[i, j] < 0 or rs[i, j] > D:
                vs[i, j] *= -1 # Reflecting BCs, ideal elastic collisions with only the wall
                collnum += 1
    colls = np.append(colls, collnum) # tracks collisions
    vssq = np.sum(vs*vs, axis=1)
    avgKE = 0.5*m*np.mean(vssq)
    T = (2.0/(kb*dim))*avgKE
    intxt.set_text(f"Frame {frame}/{ts}: Temp: {T:.2f} - Number of collisions: {collnum} - On average: {np.average(colls):.2f}")
    print(f"Frame {frame}/{ts}: Temp: {T:.2f} - Number of collisions: {collnum} - On average: {np.average(colls):.2f}", end='\r') # live collisions and temp per frame
                
    if dim == 2:
        scatter.set_offsets(rs)
    if dim == 3:
        scatter.set_offsets(rs[:, :2])
        scatter.set_3d_properties(rs[:, 2], 'z')
    return scatter, 

if dim == 2:
    ani = animation.FuncAnimation(fig, update, frames=ts, interval=20, blit=True)
if dim == 3:
    ani = animation.FuncAnimation(fig, update, frames=ts, interval=20, blit=False)

plt.close(fig) 

# save the HTML player as a file in the folder the code is in
with open("particle_animation.html", "w") as f:
    f.write(ani.to_jshtml())
HTML(ani.to_jshtml())
